# Notebook 4: SQL Business Analytics & Revenue KPIs
# Hotel Revenue Management & Dynamic Pricing Intelligence
## Objective

This notebook performs business-oriented SQL analysis on the engineered hotel booking dataset stored in PostgreSQL.

Unlike the exploratory analysis performed in the previous notebook, this phase focuses on answering business questions using SQL. The analysis is designed to simulate the work of a hotel revenue analyst by generating executive KPIs, demand insights, cancellation metrics, pricing analysis, and customer segmentation directly from the relational database.

The insights obtained in this notebook will serve as the analytical foundation for the Dynamic Pricing Engine and Executive Dashboard developed in the subsequent stages of the project.

## Dataset Information

**Database:** PostgreSQL

**Database Name:** hotel_revenue_db

**Table:** hotel_bookings

**Records:** 119,390

**Features:** 49

The dataset has already been cleaned and enriched with engineered features during the previous data preparation phase. SQL is now used to perform analytical queries directly on the production-ready database.

## SQL Workflow

The notebook is organised into the following sections:

1. Database Connection
2. Database Validation
3. Executive KPI Analysis
4. Demand Analysis
5. Cancellation Analytics
6. Revenue Analytics
7. Customer Analytics
8. Advanced SQL Analytics
9. Executive Recommendations

# 1. Database Connection
This section establishes a connection between Python and the PostgreSQL database.

All SQL queries executed throughout this notebook will run directly against the PostgreSQL database, allowing business insights to be generated without relying on intermediate CSV files.

In [1]:
# ==========================================================
# Import Libraries
# ==========================================================

import pandas as pd
import psycopg2
from sqlalchemy import create_engine

In [2]:
# ==========================================================
# PostgreSQL Connection
# ==========================================================

DB_USER = "postgres"
DB_PASSWORD = "bush1906"
DB_HOST = "localhost"
DB_PORT = "5432"
DB_NAME = "hotel_revenue_db"

In [3]:
# ==========================================================
# Create SQLAlchemy Engine
# ==========================================================

engine = create_engine(
    f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

print("Database connected successfully.")

Database connected successfully.


In [4]:
# ==========================================================
# Test Database Connection
# ==========================================================

query = """
SELECT COUNT(*) AS total_rows
FROM hotel_bookings;
"""

pd.read_sql(query, engine)

,total_rows
0,119390


# 2. Database Validation
Before performing business analysis, the imported PostgreSQL database is validated to ensure that the migration from the cleaned dataset was successful.

This validation confirms that the expected records, columns, and data quality characteristics have been preserved after loading the data into PostgreSQL.

In [5]:
# ==========================================================
# Function to Execute SQL Queries
# ==========================================================

def run_query(query):
    """
    Executes a SQL query on PostgreSQL
    and returns the result as a pandas DataFrame.
    """
    return pd.read_sql(query, engine)

### Validation Check 1

Verify that the expected number of reservation records has been imported.

In [6]:
query = """
SELECT COUNT(*) AS total_reservations
FROM hotel_bookings;
"""

pd.read_sql(query, engine)

,total_reservations
0,119390


### Validation Check 2

Verify the number of available attributes.

In [7]:
query = """
SELECT COUNT(*) AS total_columns

FROM information_schema.columns

WHERE table_name='hotel_bookings';
"""

pd.read_sql(query, engine)

,total_columns
0,49


### Validation Check 3

Check missing values in important business attributes.

In [8]:
query = """
SELECT

SUM(
CASE
WHEN country IS NULL
THEN 1
ELSE 0
END
) AS missing_country,

SUM(
CASE
WHEN children IS NULL
THEN 1
ELSE 0
END
) AS missing_children,

SUM(
CASE
WHEN agent IS NULL
THEN 1
ELSE 0
END
) AS missing_agent,

SUM(
CASE
WHEN company IS NULL
THEN 1
ELSE 0
END
) AS missing_company

FROM hotel_bookings;
"""

pd.read_sql(query, engine)

,missing_country,missing_children,missing_agent,missing_company
0,0,0,16340,112593


### Validation Check 4

Verify duplicate reservation records.

In [9]:
query = """
SELECT
COUNT(*) AS total_rows,
COUNT(DISTINCT(
hotel,
arrival_date,
departure_date,
adr,
total_guests,
booking_status
))
AS unique_rows
FROM hotel_bookings;
"""

pd.read_sql(query, engine)

,total_rows,unique_rows
0,119390,74124


### Validation Summary

The PostgreSQL database successfully preserves the cleaned reservation dataset generated during the data preparation stage.

The expected number of records and attributes are available, while the missing values remain consistent with those identified during data cleaning. No issues were introduced during the migration process, confirming that the database is ready for business analytics.

# 3. Executive KPI Analysis

Key Performance Indicators (KPIs) provide a high-level summary of hotel performance and are widely used by revenue managers to monitor occupancy, pricing, customer behaviour, and revenue generation.

This section calculates the core business metrics that will later be displayed in the executive dashboard and used by the dynamic pricing engine.

### Business Question 1

**What is the overall booking performance of the hotel network?**

This query summarizes the most important operational KPIs, including booking volume, pricing, customer behavior, and cancellation rate.

In [10]:
overall_kpi_query = """
SELECT

    COUNT(*) AS total_bookings,

    ROUND(AVG(adr),2) AS average_adr,

    ROUND(AVG(lead_time),2) AS average_lead_time,

    ROUND(AVG(stay_duration),2) AS average_stay_duration,

    ROUND(AVG(total_guests),2) AS average_guests_per_booking,

    ROUND(AVG(is_canceled) * 100,2) AS cancellation_rate

FROM hotel_bookings;
"""

run_query(overall_kpi_query)

,total_bookings,average_adr,average_lead_time,average_stay_duration,average_guests_per_booking,cancellation_rate
0,119390,101.83,104.01,3.43,1.97,37.04


### Key Findings

- The dataset contains **119,390 reservations**.
- The average ADR is **$101.83**.
- Guests book **104 days** before arrival on average.
- The average stay duration is **3.43 nights**.
- The overall cancellation rate is **37.04%**.

### Recommendation

The high cancellation rate highlights the need for demand forecasting and dynamic pricing strategies that account for booking lead time and cancellation risk.

### Business Question 2

**How does business performance differ between City Hotels and Resort Hotels?**

This analysis compares booking volume, pricing, cancellations, average stay duration, and estimated revenue across hotel types.

In [11]:
hotel_performance_query = """
SELECT

    hotel,

    COUNT(*) AS total_bookings,

    ROUND(AVG(adr),2) AS average_adr,

    ROUND(AVG(is_canceled) * 100,2) AS cancellation_rate,

    ROUND(AVG(stay_duration),2) AS average_stay_duration,

    ROUND(SUM(adr),2) AS estimated_revenue

FROM hotel_bookings

GROUP BY hotel

ORDER BY estimated_revenue DESC;
"""

run_query(hotel_performance_query)

,hotel,total_bookings,average_adr,cancellation_rate,average_stay_duration,estimated_revenue
0,City Hotel,79330,105.30,41.73,2.98,8353803.24
1,Resort Hotel,40060,94.95,27.76,4.32,3803814.36


### Key Findings

- City Hotels account for nearly **66% of total bookings**.
- City Hotels achieve higher ADR and generate significantly more revenue.
- Resort Hotels have longer average stays.
- City Hotels experience a much higher cancellation rate.

### Recommendation

Prioritize dynamic pricing for City Hotels while introducing measures to reduce cancellations. Promote longer-stay offers at Resort Hotels to improve occupancy.

### Business Question 3

**Which months contribute the highest booking demand and revenue potential?**

Understanding seasonal booking behaviour helps revenue managers prepare pricing strategies for peak and off-peak periods.

In [12]:
monthly_performance_query = """
SELECT

    arrival_month_number,

    arrival_date_month,

    COUNT(*) AS total_bookings,

    ROUND(AVG(adr),2) AS average_adr,

    ROUND(SUM(adr),2) AS estimated_revenue

FROM hotel_bookings

GROUP BY

    arrival_month_number,
    arrival_date_month

ORDER BY arrival_month_number;
"""

run_query(monthly_performance_query)

,arrival_month_number,arrival_date_month,total_bookings,average_adr,estimated_revenue
0,1,January,5929,70.36,417171.80
1,2,February,8068,73.58,593661.80
2,3,March,9794,80.68,790176.45
3,4,April,11089,100.38,1113122.58
4,5,May,11791,108.70,1281628.83
5,6,June,10939,116.67,1276277.11
6,7,July,12661,126.79,1605263.03
7,8,August,13877,140.11,1944327.60
8,9,September,10508,105.05,1103861.80
9,10,October,11160,87.91,981063.09


### Key Findings

- Demand and ADR increase steadily from **January to August**.
- **August** records the highest ADR and estimated revenue.
- Booking activity declines during the final quarter of the year.

### Recommendation

Apply surge pricing during peak months (July–August) and offer targeted promotions during low-demand months to improve occupancy.

### Business Question 4

**Which customer group contributes the greatest revenue?**

Comparing customer groups helps identify high-value guests and supports targeted pricing and marketing decisions.

In [13]:
customer_segment_query = """
SELECT

    family_type,

    COUNT(*) AS total_bookings,

    ROUND(AVG(adr),2) AS average_adr,

    ROUND(SUM(adr),2) AS estimated_revenue

FROM hotel_bookings

GROUP BY family_type

ORDER BY estimated_revenue DESC;
"""

run_query(customer_segment_query)

,family_type,total_bookings,average_adr,estimated_revenue
0,Couple,82051,98.03,8043734.53
1,Group,14578,155.75,2270492.27
2,Solo,22581,81.55,1841508.63
3,Unknown,180,10.46,1882.17


### Key Findings

- Couples contribute the largest share of bookings and revenue.
- Group bookings achieve the highest average ADR.
- Solo travelers generate lower revenue due to lower ADR.

### Recommendation

Focus marketing efforts on couples while expanding premium group packages to increase high-value bookings.

# 4. Demand Analysis

Understanding booking demand patterns is essential for effective revenue management. This section analyzes how demand varies across seasons, booking windows, and customer segments to identify opportunities for dynamic pricing and occupancy optimization.

### Business Question 1

**How does booking demand vary across different seasons?**

Seasonal demand patterns help identify peak and off-peak periods, allowing hotels to optimize pricing strategies throughout the year.

In [14]:
seasonal_demand_query = """
SELECT

    arrival_season,

    COUNT(*) AS total_bookings,

    ROUND(AVG(adr),2) AS average_adr,

    ROUND(SUM(adr),2) AS potential_room_revenue

FROM hotel_bookings

GROUP BY arrival_season

ORDER BY potential_room_revenue DESC;
"""

run_query(seasonal_demand_query)

,arrival_season,total_bookings,average_adr,potential_room_revenue
0,Summer,37477,128.77,4825867.74
1,Spring,32674,97.48,3184927.86
2,Autumn,28462,90.87,2586287.86
3,Winter,20777,75.11,1560534.14


### Key Findings

- **Summer** has the highest booking volume and ADR.
- Summer generates the highest potential room revenue.
- **Winter** has the lowest demand and ADR.

### Recommendation

Use higher pricing during summer peak demand and targeted promotions during winter to improve occupancy.

### Business Question 2

**How does booking demand vary across lead time categories?**

Understanding booking behaviour by lead time helps identify when customers typically reserve rooms and supports dynamic pricing decisions.

In [15]:
lead_time_analysis_query = """
SELECT

    lead_time_bucket,

    COUNT(*) AS total_bookings,

    ROUND(AVG(adr),2) AS average_adr,

    ROUND(AVG(is_canceled) * 100,2) AS cancellation_rate,

    ROUND(SUM(adr),2) AS potential_room_revenue

FROM hotel_bookings

GROUP BY lead_time_bucket

ORDER BY
CASE lead_time_bucket
    WHEN '0-7 Days' THEN 1
    WHEN '8-30 Days' THEN 2
    WHEN '31-90 Days' THEN 3
    WHEN '90+ Days' THEN 4
END;
"""

run_query(lead_time_analysis_query)

,lead_time_bucket,total_bookings,average_adr,cancellation_rate,potential_room_revenue
0,0-7 Days,19746,89.97,9.63,1776562.99
1,8-30 Days,18960,108.12,27.86,2049985.74
2,31-90 Days,29553,106.57,37.70,3149534.91
3,90+ Days,51131,101.34,50.65,5181533.96


### Key Findings

- **90+ day bookings** have the highest volume but also the highest cancellation rate (**50.65%**).
- Short-lead bookings have the lowest cancellation rate (**9.63%**).
- ADR is highest for bookings made **8–30 days** in advance.

### Recommendation

Use stricter cancellation policies or deposits for long-lead bookings while protecting short-lead demand with flexible booking options.

### Business Question 3

**Which day of the week receives the highest number of arrivals?**

Daily arrival patterns help hotels plan staffing, housekeeping, and operational resources more effectively.

In [17]:
arrival_day_query = """
SELECT

    arrival_day_name,

    COUNT(*) AS total_arrivals,

    ROUND(AVG(adr),2) AS average_adr

FROM hotel_bookings

GROUP BY arrival_day_name

ORDER BY
CASE arrival_day_name
    WHEN 'Monday' THEN 1
    WHEN 'Tuesday' THEN 2
    WHEN 'Wednesday' THEN 3
    WHEN 'Thursday' THEN 4
    WHEN 'Friday' THEN 5
    WHEN 'Saturday' THEN 6
    WHEN 'Sunday' THEN 7
END;
"""

run_query(arrival_day_query)

,arrival_day_name,total_arrivals,average_adr
0,Monday,18171,102.75
1,Tuesday,13999,100.64
2,Wednesday,16139,98.58
3,Thursday,19254,98.68
4,Friday,19631,105.52
5,Saturday,18055,102.13
6,Sunday,14141,104.33


### Key Findings

- **Friday** has the highest number of arrivals.
- **Tuesday** has the lowest arrival volume.
- Sunday has the highest ADR despite lower arrival volume.

### Recommendation

Increase operational capacity around Friday arrivals and consider targeted offers to strengthen weaker weekdays.

### Business Question 4

**Which quarters generate the highest booking demand and revenue?**

Quarterly demand analysis helps management plan pricing strategies and allocate resources across the year.

In [16]:
quarterly_analysis_query = """
SELECT

    arrival_quarter,

    COUNT(*) AS total_bookings,

    ROUND(AVG(adr),2) AS average_adr,

    ROUND(SUM(adr),2) AS potential_room_revenue

FROM hotel_bookings

GROUP BY arrival_quarter

ORDER BY arrival_quarter;
"""

run_query(quarterly_analysis_query)

,arrival_quarter,total_bookings,average_adr,potential_room_revenue
0,Q1,23791,75.70,1801010.05
1,Q2,33819,108.55,3671028.52
2,Q3,37046,125.61,4653452.43
3,Q4,24734,82.16,2032126.60


### Key Findings

- **Q3** has the highest booking volume and ADR.
- Q3 also generates the highest potential room revenue.
- **Q1** has the lowest demand, ADR, and revenue.

### Recommendation

Apply peak-season pricing during Q3 and use promotions during Q1 to balance demand across the year.

# 5. Cancellation Analytics

Understanding cancellation patterns helps the hotel identify high-risk bookings, reduce revenue loss, and design better cancellation policies.

### Business Question 1

**Which lead-time groups have the highest cancellation risk?**

Identifying high-risk booking windows can help the hotel apply appropriate cancellation policies and reduce revenue uncertainty.

In [18]:
cancellation_lead_query = """
SELECT

    lead_time_bucket,

    COUNT(*) AS total_bookings,

    SUM(is_canceled) AS canceled_bookings,

    ROUND(AVG(is_canceled) * 100, 2) AS cancellation_rate

FROM hotel_bookings

GROUP BY lead_time_bucket

ORDER BY cancellation_rate DESC;
"""

run_query(cancellation_lead_query)

,lead_time_bucket,total_bookings,canceled_bookings,cancellation_rate
0,90+ Days,51131,25898,50.65
1,31-90 Days,29553,11141,37.70
2,8-30 Days,18960,5283,27.86
3,0-7 Days,19746,1902,9.63


### Key Findings

- Bookings made **90+ days ahead** have the highest cancellation rate at **50.65%**.
- Short-lead bookings have the lowest cancellation rate at **9.63%**.
- Cancellation risk increases substantially with longer lead times.

### Recommendation

Use stricter cancellation policies or deposits for long-lead bookings to reduce revenue uncertainty.

### Business Question 2

**Which deposit types are associated with the highest cancellation rates?**

Understanding cancellation behaviour across deposit policies can help the hotel evaluate the effectiveness of its payment and cancellation policies.

In [19]:
cancellation_deposit_query = """
SELECT

    deposit_type,

    COUNT(*) AS total_bookings,

    SUM(is_canceled) AS canceled_bookings,

    ROUND(AVG(is_canceled) * 100, 2) AS cancellation_rate

FROM hotel_bookings

GROUP BY deposit_type

ORDER BY cancellation_rate DESC;
"""

run_query(cancellation_deposit_query)

,deposit_type,total_bookings,canceled_bookings,cancellation_rate
0,Non Refund,14587,14494,99.36
1,No Deposit,104641,29694,28.38
2,Refundable,162,36,22.22


### Key Findings

- **Non-Refund** bookings have an extremely high cancellation rate of **99.36%**.
- **No Deposit** bookings have a cancellation rate of **28.38%**.
- Refundable bookings have the lowest rate at **22.22%**, but represent a very small number of bookings.

### Recommendation

Investigate the unusually high cancellation rate for Non-Refund bookings and review how this deposit category is being recorded or applied.

### Business Question 3

**Which hotel type experiences more cancellations?**

Comparing cancellation behaviour between hotel types helps identify where revenue risk is concentrated.

In [20]:
cancellation_hotel_query = """
SELECT

    hotel,

    COUNT(*) AS total_bookings,

    SUM(is_canceled) AS canceled_bookings,

    ROUND(AVG(is_canceled) * 100, 2) AS cancellation_rate

FROM hotel_bookings

GROUP BY hotel

ORDER BY cancellation_rate DESC;
"""

run_query(cancellation_hotel_query)

,hotel,total_bookings,canceled_bookings,cancellation_rate
0,City Hotel,79330,33102,41.73
1,Resort Hotel,40060,11122,27.76


### Key Findings

- **City Hotels** have a higher cancellation rate (**41.73%**) than Resort Hotels (**27.76%**).
- City Hotels also have a much larger booking volume.
- Cancellation risk is therefore more concentrated in City Hotels.

### Recommendation

Prioritize cancellation-reduction strategies for City Hotels, particularly for high-risk booking segments.

### Business Question 4

**Does repeated-guest status affect cancellation behaviour?**

Comparing new and repeat guests can help determine whether customer loyalty is associated with more reliable bookings.

In [21]:
cancellation_repeat_query = """
SELECT

    CASE
        WHEN is_repeated_guest = 1 THEN 'Repeat Guest'
        ELSE 'New Guest'
    END AS guest_type,

    COUNT(*) AS total_bookings,

    SUM(is_canceled) AS canceled_bookings,

    ROUND(AVG(is_canceled) * 100, 2) AS cancellation_rate

FROM hotel_bookings

GROUP BY is_repeated_guest

ORDER BY cancellation_rate DESC;
"""

run_query(cancellation_repeat_query)

,guest_type,total_bookings,canceled_bookings,cancellation_rate
0,New Guest,115580,43672,37.79
1,Repeat Guest,3810,552,14.49


### Key Findings

- **New Guests** have a cancellation rate of **37.79%**.
- **Repeat Guests** have a much lower rate of **14.49%**.
- Repeat guests appear to be significantly more reliable.

### Recommendation

Strengthen loyalty programs and targeted retention campaigns to increase repeat bookings and reduce cancellation risk.

# 6. Customer & Booking Behaviour

This section analyzes customer types, booking channels, market segments, and guest booking behaviour to understand who is driving hotel demand and how they interact with the booking process.

## Business Question 1

**Which customer types generate the most bookings?**

Understanding customer composition helps management identify the hotel's primary customer segments.

In [22]:
customer_type_query = """
SELECT

    customer_type,

    COUNT(*) AS total_bookings,

    ROUND(AVG(adr), 2) AS average_adr,

    ROUND(SUM(adr), 2) AS estimated_revenue

FROM hotel_bookings

GROUP BY customer_type

ORDER BY total_bookings DESC;
"""

run_query(customer_type_query)

,customer_type,total_bookings,average_adr,estimated_revenue
0,Transient,89613,107.01,9589811.59
1,Transient-Party,25124,86.08,2162780.78
2,Contract,4076,87.55,356852.32
3,Group,577,83.49,48172.91


### Key Findings

- **Transient customers** generate the majority of bookings and revenue.
- They also have the highest ADR among the major customer types.
- Group bookings have a much higher ADR but very low booking volume.

### Recommendation

Maintain focus on transient customers while developing targeted strategies to attract more high-value group bookings.

## Business Question 2

**Which booking channels generate the most reservations?**

Comparing distribution channels helps the hotel understand where its bookings are coming from and where channel performance is concentrated.

In [23]:
distribution_channel_query = """
SELECT

    distribution_channel,

    COUNT(*) AS total_bookings,

    ROUND(AVG(adr), 2) AS average_adr,

    ROUND(SUM(adr), 2) AS estimated_revenue

FROM hotel_bookings

GROUP BY distribution_channel

ORDER BY total_bookings DESC;
"""

run_query(distribution_channel_query)

,distribution_channel,total_bookings,average_adr,estimated_revenue
0,TA/TO,97870,103.29,10109272.81
1,Direct,14645,106.65,1561962.22
2,Corporate,6677,69.33,462884.39
3,GDS,193,120.55,23266.98
4,Undefined,5,46.24,231.20


### Key Findings

- **TA/TO** generates the overwhelming majority of bookings and revenue.
- Direct bookings have a higher ADR than TA/TO bookings.
- Corporate bookings contribute a smaller share of total revenue.

### Recommendation

Maintain strong relationships with TA/TO channels while encouraging more direct bookings to potentially improve revenue per booking and reduce channel dependency.

## Business Question 3

**Which market segments contribute the most bookings and revenue?**

Market-segment analysis helps management identify the most important customer acquisition segments.

In [24]:
market_segment_query = """
SELECT

    market_segment,

    COUNT(*) AS total_bookings,

    ROUND(AVG(adr), 2) AS average_adr,

    ROUND(SUM(adr), 2) AS estimated_revenue

FROM hotel_bookings

GROUP BY market_segment

ORDER BY estimated_revenue DESC;
"""

run_query(market_segment_query)

,market_segment,total_bookings,average_adr,estimated_revenue
0,Online TA,56477,117.20,6618938.52
1,Offline TA/TO,24219,87.35,2115645.48
2,Groups,19811,79.48,1574567.82
3,Direct,12606,115.45,1455301.88
4,Corporate,5295,69.36,367255.65
5,Aviation,237,100.14,23733.68
6,Complementary,743,2.89,2144.57
7,Undefined,2,15.00,30.00


### Key Findings

- **Online TA** is the largest market segment by both bookings and estimated revenue.
- Group bookings have a relatively high booking volume but lower ADR.
- Direct bookings have a higher ADR than Offline TA/TO bookings.

### Recommendation

Continue investing in high-performing online channels while developing strategies to increase direct bookings and improve their contribution.

## Business Question 4

**How does the number of special requests relate to booking behaviour?**

Special requests can indicate guest engagement and planning needs, which may help identify differences between booking types.

In [25]:
special_requests_query = """
SELECT

    total_of_special_requests,

    COUNT(*) AS total_bookings,

    ROUND(AVG(adr), 2) AS average_adr,

    ROUND(AVG(is_canceled) * 100, 2) AS cancellation_rate

FROM hotel_bookings

GROUP BY total_of_special_requests

ORDER BY total_of_special_requests;
"""

run_query(special_requests_query)

,total_of_special_requests,total_bookings,average_adr,cancellation_rate
0,0,70318,94.96,47.72
1,1,33226,108.46,22.02
2,2,12969,117.03,22.10
3,3,2497,123.85,17.86
4,4,340,130.57,10.59
5,5,40,127.75,5.00


### Key Findings

- Most bookings have **no special requests**.
- Bookings with special requests generally have higher ADR.
- Cancellation rate decreases as the number of special requests increases.

### Recommendation

Use special-request activity as a possible indicator of guest engagement and booking commitment when developing customer segmentation strategies.

# 7. Revenue & Pricing Analysis

This section analyzes booking revenue, pricing patterns, and the factors associated with higher room rates.

## Business Question 1

**How does revenue and ADR vary by hotel type?**

Comparing hotel types helps management understand which property type generates greater booking value.

In [26]:
hotel_revenue_query = """
SELECT

    hotel,

    COUNT(*) AS total_bookings,

    ROUND(AVG(adr), 2) AS average_adr,

    ROUND(SUM(adr), 2) AS estimated_revenue,

    ROUND(SUM(adr) / COUNT(*), 2) AS revenue_per_booking

FROM hotel_bookings

GROUP BY hotel

ORDER BY estimated_revenue DESC;
"""

run_query(hotel_revenue_query)

,hotel,total_bookings,average_adr,estimated_revenue,revenue_per_booking
0,City Hotel,79330,105.30,8353803.24,105.30
1,Resort Hotel,40060,94.95,3803814.36,94.95


### Key Findings

- City Hotels generate higher revenue than Resort Hotels.
- City Hotels have a higher ADR of **$105.30** compared with **$94.95** for Resort Hotels.
- Higher booking volume and ADR make City Hotels the stronger revenue contributor.

### Recommendation

Focus on maintaining City Hotel demand while using targeted pricing strategies to improve Resort Hotel revenue.

## Business Question 2

**How does pricing change across the seasons?**

Seasonal pricing analysis helps management identify periods of strong demand and opportunities for revenue optimization.

In [27]:
seasonal_pricing_query = """
SELECT

    arrival_season,

    COUNT(*) AS total_bookings,

    ROUND(AVG(adr), 2) AS average_adr,

    ROUND(SUM(adr), 2) AS estimated_revenue

FROM hotel_bookings

GROUP BY arrival_season

ORDER BY average_adr DESC;
"""

run_query(seasonal_pricing_query)

,arrival_season,total_bookings,average_adr,estimated_revenue
0,Summer,37477,128.77,4825867.74
1,Spring,32674,97.48,3184927.86
2,Autumn,28462,90.87,2586287.86
3,Winter,20777,75.11,1560534.14


### Key Findings

- **Summer** has the highest ADR at **$128.77** and the highest estimated revenue.
- **Winter** has the lowest ADR at **$75.11**.
- Pricing and revenue vary significantly across seasons.

### Recommendation

Use dynamic pricing during high-demand seasons and attractive offers during lower-demand periods to balance occupancy and revenue.

## Business Question 3

**Which customer groups generate the highest booking value?**

Comparing ADR and revenue across customer groups helps identify high-value customer segments.

In [28]:
customer_value_query = """
SELECT

    family_type,

    COUNT(*) AS total_bookings,

    ROUND(AVG(adr), 2) AS average_adr,

    ROUND(SUM(adr), 2) AS estimated_revenue

FROM hotel_bookings

GROUP BY family_type

ORDER BY average_adr DESC;
"""

run_query(customer_value_query)

,family_type,total_bookings,average_adr,estimated_revenue
0,Group,14578,155.75,2270492.27
1,Couple,82051,98.03,8043734.53
2,Solo,22581,81.55,1841508.63
3,Unknown,180,10.46,1882.17


### Key Findings

- **Group bookings** have the highest ADR at **$155.75**.
- Couples generate the highest estimated revenue because of their much larger booking volume.
- Solo bookings have lower ADR and revenue compared with couples and groups.

### Recommendation

Target group customers with suitable packages while continuing to retain the large and stable couple segment.

## Business Question 4

**How does booking lead time relate to pricing?**

Understanding the relationship between booking timing and ADR can help management evaluate pricing and advance-booking strategies.

In [29]:
lead_time_pricing_query = """
SELECT

    lead_time_bucket,

    COUNT(*) AS total_bookings,

    ROUND(AVG(adr), 2) AS average_adr,

    ROUND(SUM(adr), 2) AS estimated_revenue

FROM hotel_bookings

GROUP BY lead_time_bucket

ORDER BY
    CASE lead_time_bucket
        WHEN '0-7 Days' THEN 1
        WHEN '8-30 Days' THEN 2
        WHEN '31-90 Days' THEN 3
        WHEN '90+ Days' THEN 4
    END;
"""

run_query(lead_time_pricing_query)

,lead_time_bucket,total_bookings,average_adr,estimated_revenue
0,0-7 Days,19746,89.97,1776562.99
1,8-30 Days,18960,108.12,2049985.74
2,31-90 Days,29553,106.57,3149534.91
3,90+ Days,51131,101.34,5181533.96


### Key Findings

- Bookings made **8–30 days in advance** have the highest ADR at **$108.12**.
- Very early bookings (90+ days) have a slightly lower ADR of **$101.34** despite having the highest booking volume.
- Last-minute bookings (0–7 days) have the lowest ADR at **$89.97**.
- The 90+ day segment generates the highest estimated revenue because of its large booking volume.

### Recommendation

Use advance-booking promotions to secure demand early, while using flexible pricing to capture higher rates from customers booking closer to arrival.

# 8. Advanced Business Analysis

This section uses advanced SQL techniques to identify performance rankings, trends, and comparisons that are difficult to answer with basic GROUP BY queries alone.

## Business Question 1

**How does each hotel's revenue compare with the overall hotel network?**

A CTE allows us to calculate hotel-level revenue first and then compare it with the overall network revenue.

In [30]:
hotel_revenue_comparison_query = """
WITH hotel_revenue AS (

    SELECT
        hotel,
        ROUND(SUM(adr), 2) AS hotel_revenue

    FROM hotel_bookings

    GROUP BY hotel
),

overall_revenue AS (

    SELECT
        ROUND(SUM(adr), 2) AS total_network_revenue

    FROM hotel_bookings
)

SELECT

    h.hotel,

    h.hotel_revenue,

    o.total_network_revenue,

    ROUND(
        h.hotel_revenue * 100.0 / o.total_network_revenue,
        2
    ) AS revenue_share_percent

FROM hotel_revenue h

CROSS JOIN overall_revenue o

ORDER BY hotel_revenue DESC;
"""

run_query(hotel_revenue_comparison_query)

,hotel,hotel_revenue,total_network_revenue,revenue_share_percent
0,City Hotel,8353803.24,12157617.6,68.71
1,Resort Hotel,3803814.36,12157617.6,31.29


### Key Findings

- City Hotel contributes **68.71%** of total network revenue.
- Resort Hotel contributes **31.29%**.
- City Hotel is the stronger revenue contributor.

### Recommendation

Continue strengthening City Hotel performance while identifying opportunities to increase Resort Hotel's revenue contribution.

## Business Question 2

**Which months generate the highest revenue?**

Ranking monthly revenue helps management identify the strongest revenue periods.

In [31]:
monthly_revenue_rank_query = """
WITH monthly_revenue AS (

    SELECT

        arrival_month_number,
        arrival_date_month,

        ROUND(SUM(adr), 2) AS monthly_revenue

    FROM hotel_bookings

    GROUP BY
        arrival_month_number,
        arrival_date_month
)

SELECT

    arrival_date_month,

    monthly_revenue,

    RANK() OVER (
        ORDER BY monthly_revenue DESC
    ) AS revenue_rank

FROM monthly_revenue

ORDER BY revenue_rank;
"""

run_query(monthly_revenue_rank_query)

,arrival_date_month,monthly_revenue,revenue_rank
0,August,1944327.60,1
1,July,1605263.03,2
2,May,1281628.83,3
3,June,1276277.11,4
4,April,1113122.58,5
5,September,1103861.80,6
6,October,981063.09,7
7,March,790176.45,8
8,February,593661.80,9
9,December,549700.54,10


### Key Findings

- **August** ranks first with approximately **$1.94M** in potential room revenue.
- **July** ranks second at approximately **$1.61M**.
- **January** has the lowest potential room revenue at approximately **$417K**.
- Revenue is strongly concentrated in the summer months.

### Recommendation

Use higher pricing during July–August peak demand and targeted promotions during low-revenue months such as January to improve overall revenue performance.

## Business Question 3

**How does each month's revenue contribute to cumulative annual demand?**

A running total helps management understand how revenue accumulates throughout the year.

In [32]:
monthly_running_revenue_query = """
WITH monthly_revenue AS (

    SELECT

        arrival_date_year,
        arrival_month_number,
        arrival_date_month,

        ROUND(SUM(adr), 2) AS monthly_revenue

    FROM hotel_bookings

    GROUP BY
        arrival_date_year,
        arrival_month_number,
        arrival_date_month
)

SELECT

    arrival_date_year,

    arrival_date_month,

    monthly_revenue,

    ROUND(
        SUM(monthly_revenue) OVER (
            PARTITION BY arrival_date_year
            ORDER BY arrival_month_number
        ),
        2
    ) AS cumulative_revenue

FROM monthly_revenue

ORDER BY
    arrival_date_year,
    arrival_month_number;
"""

run_query(monthly_running_revenue_query)

,arrival_date_year,arrival_date_month,monthly_revenue,cumulative_revenue
0,2015,July,271588.06,271588.06
1,2015,August,411934.11,683522.17
2,2015,September,484902.64,1168424.81
3,2015,October,391084.63,1559509.44
4,2015,November,141757.79,1701267.23
5,2015,December,216311.39,1917578.62
6,2016,January,145597.69,145597.69
7,2016,February,272767.93,418365.62
8,2016,March,381430.43,799796.05
9,2016,April,482651.90,1282447.95


### Key Findings

- Revenue accumulates steadily throughout each year, with stronger growth during the middle of the year.
- **August 2016** reached approximately **$3.67M** in cumulative revenue by that point in the year.
- **August 2017** reached approximately **$4.66M**, already exceeding the same point in 2016.
- The results indicate stronger revenue performance in 2017 compared with the previous years.

### Recommendation

Use cumulative revenue tracking to monitor in-year performance against previous years and identify periods where pricing or promotional adjustments may be required.

## Business Question 4

**Which market segments perform best within each hotel type?**

Comparing segments within each hotel helps identify the strongest customer acquisition opportunities for each property.

In [33]:
hotel_segment_ranking_query = """
WITH segment_performance AS (

    SELECT

        hotel,
        market_segment,

        COUNT(*) AS total_bookings,

        ROUND(AVG(adr), 2) AS average_adr,

        ROUND(SUM(adr), 2) AS estimated_revenue

    FROM hotel_bookings

    GROUP BY
        hotel,
        market_segment
)

SELECT

    hotel,
    market_segment,
    total_bookings,
    average_adr,
    estimated_revenue,

    ROW_NUMBER() OVER (
        PARTITION BY hotel
        ORDER BY estimated_revenue DESC
    ) AS segment_rank

FROM segment_performance

ORDER BY
    hotel,
    segment_rank;
"""

run_query(hotel_segment_ranking_query)

,hotel,market_segment,total_bookings,average_adr,estimated_revenue,segment_rank
0,City Hotel,Online TA,38748,118.92,4607894.08,1
1,City Hotel,Offline TA/TO,16747,93.02,1557766.75,2
2,City Hotel,Groups,13975,84.92,1186783.34,3
3,City Hotel,Direct,6093,119.48,727989.70,4
4,City Hotel,Corporate,2986,83.12,248196.26,5
5,City Hotel,Aviation,237,100.14,23733.68,6
6,City Hotel,Complementary,542,2.60,1409.43,7
7,City Hotel,Undefined,2,15.00,30.00,8
8,Resort Hotel,Online TA,17729,113.43,2011044.44,1
9,Resort Hotel,Direct,6513,111.67,727312.18,2


### Key Findings

- **Online TA** ranks first for both City and Resort Hotels.
- Online TA contributes approximately **$4.61M** for City Hotels and **$2.01M** for Resort Hotels.
- Direct bookings rank second for Resort Hotels and have a higher ADR than most other segments.
- Complementary bookings contribute very little revenue.

### Recommendation

Maintain strong Online TA performance while increasing direct bookings to reduce channel dependency and potentially improve revenue per booking.

# 9. Executive Recommendations

The SQL analysis identified several patterns that can directly inform the hotel's revenue-management and dynamic-pricing strategy.

### 1. Seasonal Pricing

Demand and ADR are strongest during summer, particularly July and August.

**Action:** Increase prices during high-demand periods and use targeted promotions during low-demand periods such as winter.

### 2. Cancellation Risk

Long-lead bookings and City Hotel reservations show higher cancellation risk, creating uncertainty in future occupancy.

**Action:** Incorporate cancellation probability into demand forecasting and pricing decisions, particularly for bookings made far in advance.

### 3. Hotel-Level Pricing

City Hotels contribute approximately **68.71% of network revenue**, making them the primary revenue contributor.

**Action:** Prioritize City Hotel pricing optimization while identifying opportunities to improve Resort Hotel revenue.

### 4. Customer & Channel Strategy

Online TA dominates booking volume, while direct and group bookings can achieve higher ADR in specific segments.

**Action:** Maintain strong high-volume channels while encouraging direct and high-value group bookings.

### 5. Dynamic Pricing Opportunity

The analysis shows that demand, lead time, seasonality, hotel type, customer segment, and cancellation risk vary significantly across bookings.

**Action:** These variables can be incorporated into a dynamic pricing engine that adjusts recommended room prices according to expected demand and booking conditions.

# Conclusion

The SQL analysis transformed reservation-level data into actionable revenue-management insights.

The results highlight clear differences in demand, pricing, cancellation behaviour, customer segments, and booking channels. These findings provide the foundation for moving from descriptive analysis toward predictive and prescriptive revenue optimization.

The next phase will use these insights to develop a **Dynamic Pricing Engine** capable of recommending price adjustments under different demand conditions and evaluating the potential revenue impact of dynamic pricing.